In [ ]:
#@title 按這裡開始（先按 ▶）
print("✅ W17 出發！本週目標：跑 baseline、調一個參數、存模型，再用 gradio 做網頁 Demo")
print("三段節奏：跑通 40 分 → 調參 40 分 → Demo 40 分，時間到就停止調參")
print("本週要自己補三個空：predict 的機率、最大的那一類、gradio 的輸入元件")

# W17　訓練、存模型與 gradio Demo（電腦教室版）

**今天結束前一定要有兩樣東西：一個點得動的 Demo，和一支 60 秒備援影片。**

**三段節奏（時間是硬性的）**
1. 第一段 40 分鐘：把 baseline 跑出一個數字。
2. 第二段 40 分鐘：一次只改一個參數，每次都填調參紀錄表。
3. 第三段 40 分鐘：存模型、寫 `predict()`、開 gradio。**剩 40 分鐘一律停止調參。**

**不同題目要改哪裡**
- 感測／數值：照著跑，讀你們的 `data.csv`。
- 文字分類：把特徵換成第 10 週的 TF-IDF 向量。
- 影像／聲音（Teachable Machine 匯出）：第 1 到 3 格跳過，
  第 4、5 格改用 `load_model` 讀 `keras_model.h5`；但混淆矩陣還是要自己算一次。

**開始之前**：功能表「檔案 → 在雲端硬碟中儲存副本」。

**想避開最後全班同時安裝的塞車**：現在就可以把最後一格的第一行
`!pip install gradio -q` 複製到這裡先跑起來。

### 示範資料：還沒有自己的 `data.csv` 就先跑這一格

**有自己的 `data.csv` 就跳過這一格**（跑了也不會覆蓋你們的檔案）。

沒有的人先用第 14 週的活動辨識資料做一份示範特徵表，
兩欄特徵 `m`（平均）、`s`（標準差）加一欄 `label`，先把整條流程走完。

**寫對了會看到什麼**：印出「已建立示範 data.csv： NN 筆」，
或是「找到你們自己的 data.csv」。

**你們自己的 `data.csv` 要長什麼樣**：一列一筆資料，
特徵各佔一欄，答案那一欄的欄名一定要叫 `label`。

**用示範資料時分數會全部是 1.0**，因為靜止與走路差太多、太好分了。
那不是你調得好，也不能寫進作業——換成你們自己的資料才有得比。

In [ ]:
#@title 示範資料（投影片未含，執行所需）
import os, pandas as pd, numpy as np      # ←投影片未含，執行所需
RAW = "https://raw.githubusercontent.com/myliao2007/stust-course-1151/main/ai-intro-pc/data/"
if os.path.exists("data.csv"):
    print("找到你們自己的 data.csv，示範資料不會覆蓋它")
else:
    t = pd.read_csv(RAW + "demo_motion.csv").iloc[:, :4]
    t.columns = ["t", "x", "y", "z"]
    t["a"] = np.sqrt(t.x**2 + t.y**2 + t.z**2)
    t["label"] = np.where(t.t < 60, "still", "walk")
    g = t.groupby([t.label, t.index // 100])
    d = pd.DataFrame({"m": g.a.mean(), "s": g.a.std()}).dropna().reset_index()
    d[["m", "s", "label"]].to_csv("data.csv", index=False)
    print("已建立示範 data.csv：", len(d), "筆（只是先讓流程跑得動）")

### 第 1 格：跑出 baseline

**這一格要做什麼**：讀資料、切訓練與測試、用預設參數跑一次交叉驗證。沒有空格。

**寫對了會看到什麼**：印出「baseline 驗證分數 = 0.xxx」，
以及訓練幾筆、測試幾筆。

**這個數字就是基準**，之後每一次改動都要回來跟它比。沒有基準的調參等於瞎猜。

`stratify=y` 是讓訓練與測試裡每一類的比例一樣，
資料不平衡的時候特別重要（第 15 週講過）。

In [ ]:
#@title 第 1 格：baseline
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
df = pd.read_csv("data.csv")
X = df.drop(columns=["label"])
y = df["label"]
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y)
base = KNeighborsClassifier()
sc = cross_val_score(base, Xtr, ytr, cv=3)
print("baseline 驗證分數 =", round(sc.mean(), 3))
print("訓練", len(Xtr), "筆　測試", len(Xte), "筆")

### 第 2 格：一次只調一個參數

**這一格要做什麼**：掃四個 `k` 值，挑分數最高的那個。沒有空格。

**寫對了會看到什麼**：四行 `k = ... 驗證分數 = ...`，
最後印出「最後選的設定： KNeighborsClassifier(n_neighbors=...)」。

**每跑一次就抄進調參紀錄表**（第幾次、改了什麼、驗證分數、要不要留），
不要只留最後一次——那張表直接截圖放進簡報，是技術分最好拿的一塊。

**一次只改一個地方**，否則說不出是誰的功勞。

In [ ]:
#@title 第 2 格：調參
best, best_sc = None, 0
for k in [3, 5, 9, 15]:
    m = KNeighborsClassifier(n_neighbors=k)
    sc = cross_val_score(m, Xtr, ytr, cv=3).mean()
    print("k =", k, "驗證分數 =", round(sc, 3))
    if sc > best_sc:
        best, best_sc = m, sc
best.fit(Xtr, ytr)
print("最後選的設定：", best)

### 第 3 格：測試分數與混淆矩陣

**整份專題只跑這一次。跑完就不要再回頭調參數了**——
再回去調就等於偷看答案，測試分數會失去意義。

**這一格要做什麼**：印出分類報告、畫混淆矩陣並存成 PNG。沒有空格。

**寫對了會看到什麼**：一份 precision／recall／f1 的報告，
一張方格圖，以及檔案區多出 `confusion.png`。

**圖上中文變方框**：Colab 沒有中文字型，類別名稱先改成英文，
或圖上不寫中文——這一點第 11、12 週都提過。

In [ ]:
#@title 第 3 格：測試分數與混淆矩陣
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
pred = best.predict(Xte)
print(classification_report(yte, pred))
cm = confusion_matrix(yte, pred, labels=best.classes_)
plt.imshow(cm)
plt.xticks(range(len(best.classes_)), best.classes_, rotation=45)
plt.yticks(range(len(best.classes_)), best.classes_)
plt.colorbar()
plt.savefig("confusion.png", dpi=150, bbox_inches="tight")
plt.show()

### 第 4 格：存模型並自己寫 `predict()`

gradio 只是一層介面，真正要自己寫的是「一筆資料進來，怎麼變成答案」。

**這一格要做什麼**：補兩行。
- 第一行：算出 `clf` 對這一筆的**機率**。
  sklearn 一次吃一整批，所以要把 `features` 包成 `[features]` 再取第 0 筆。
- 第二行：機率最大的那個**類別名稱**。
  `classes_` 是類別清單，`argmax()` 給你最大那個的位置。

**為什麼用 `predict_proba` 而不是 `predict`**：有把握度才看得出模型在猶豫，
Demo 時說服力差很多。

**寫對了會看到什麼**：印出類似 `{'類別': 'walk', '把握度': 0.83}`。
數字要換成你們自己的特徵（示範資料是兩欄，所以是兩個數字）。

**兩個常見狀況**：忘了外面那層中括號會噴 `Reshape your data`；
用 list 呼叫在 DataFrame 上訓練的模型會跳「X does not have valid feature names」
的黃色警告，那只是提醒不是錯誤。

In [ ]:
#@title 第 4 格：存模型＋自己寫 predict()
import joblib
joblib.dump(best, "model.joblib")        # 存成檔案
clf = joblib.load("model.joblib")        # 之後載入就好

def predict(features):
    """features 是一個 list，例如 [1.2, 0.8]"""
    p = ____        # ← 自己寫：clf 對 [features] 的機率
    top = ____      # ← 自己寫：機率最大的那個類別
    return {"類別": top, "把握度": round(float(max(p)), 3)}

print(predict([1.2, 0.8]))

### 第 5 格：用 gradio 做網頁 Demo

**這一格要做什麼**：補上輸入元件——一個文字輸入框。

**寫對了會看到什麼**：先印出一段安裝訊息，接著出現一個內嵌的操作介面，
以及一行 `https://xxxxx.gradio.live` 的**公開連結**。

**上線前的五個檢查**
1. 先在儲存格直接呼叫 `predict([...])`，確定不會報錯再開介面。
2. 按下這一格，等它印出 `gradio.live` 開頭的連結。
3. 自己先用另一個分頁開連結，輸入一筆看回傳對不對。
4. 把連結傳給隔壁組，請他們用自己的資料測三次。
5. 用手機錄 60 秒操作成功的畫面，存進共用資料夾當**備援影片**。

> ⚠️ **公開連結只有 72 小時效期**，而且是從 Colab 的執行階段長出來的：
> 分頁一關、執行階段一斷，連結就死了。
> **第 18 週發表當天一定要現場重跑這一格拿一條新連結。**

**教室網路擋掉 share 的話**：`share=True` 第一次會下載一個小程式，被擋就會失敗；
此時改成 `demo.launch()`，用 Colab 頁面內嵌的畫面一樣可以現場 Demo。

In [ ]:
#@title 第 5 格：gradio 網頁 Demo
!pip install gradio -q
import gradio as gr
def predict_ui(text):
    xs = [float(v) for v in text.split(",")]
    r = predict(xs)
    return f"{r['類別']}（把握度 {r['把握度']}）"
demo = gr.Interface(
    fn=predict_ui,
    inputs=____,          # ← 自己寫：一個文字輸入框
    outputs=gr.Textbox(label="模型的答案"),
    title="我們這一組的 Demo",
    description="輸入特徵值，用逗號分隔")
demo.launch(share=True)

### 收工：延伸挑戰與第 18 週的準備

- **A**：用 `gr.Interface` 的 `examples` 參數放三筆範例輸入，別人一點就能試。
- **B**（最有價值）：把連結給隔壁組，請他們用**自己的資料**測三次，記下錯在哪。
  別組一測通常會露出你們自己測不出來的問題，寫進簡報非常加分。
- **C**：說出你們的模型最常把哪兩類搞混，用混淆矩陣指出來，並說要怎麼補。

**第 18 週發表前一定要做完的五件事**
1. 混淆矩陣與調參紀錄表存成圖，貼進簡報。
2. 錄一支 60 秒備援影片。
3. 用教室電腦投影實際試一次（解析度、字級、分頁都先試過）。
4. 把 gradio 連結做成短網址或 QR，別組才好用手機開。
5. 模型檔、.ipynb、簡報、影片全部放進小組共用資料夾。

In [ ]:
#@title 收工檢查（直接按 ▶）
print("本週要交：調參紀錄表（至少五列）、混淆矩陣圖 confusion.png")
print("model.joblib 與寫完的 .ipynb，以及 gradio Demo 連結或 60 秒影片")
print("檔名：AI導論_W17_學號_姓名")
print("提醒：連結只有 72 小時，發表當天要現場重跑最後一格拿新連結")

---

<details>
<summary>參考解（三個空格都自己試過再打開）</summary>

```python
# 第 4 格
    p = clf.predict_proba([features])[0]
    top = clf.classes_[p.argmax()]

# 第 5 格
    inputs=gr.Textbox(label="輸入特徵"),
```

為什麼是這樣寫：

- sklearn 的 `predict_proba()` 一次吃一整批資料，
  所以單筆要包成 `[features]`，回傳的也是一整批，要再取 `[0]`。
- `p` 是每一類的機率（順序照 `clf.classes_`），
  `p.argmax()` 給的是「最大值在第幾個位置」，
  拿這個位置去 `clf.classes_` 裡查，才會得到類別的名字。
- `gr.Textbox()` 是文字輸入框；影像組換成 `gr.Image(type="pil")`，
  聲音組換成 `gr.Audio(type="filepath")`，`predict_ui()` 的內容也要跟著改。

</details>